In [12]:
import requests
import os
import time
import datetime
import gzip
import shutil

base_url = os.environ['CD2_BASE_URL']
client_id = os.environ['CD2_CLIENT_ID']
client_secret = os.environ['CD2_CLIENT_SECRET']

auth_url = f"{base_url}/ids/auth/login"
payload={'grant_type': 'client_credentials'}


In [ ]:

r = requests.post(
    auth_url, 
    data=payload, 
    auth=(client_id, client_secret))
if r.status_code == 200:
    respons = r.json()
    access_token = respons['access_token']
    print("Henta access_token OK")
else:
    print(f"Klarte ikkje å skaffe access_token, feil {r.status_code}")

In [9]:
def hent_filar(innfil, n):
    requesturl = f"{base_url}/dap/object/url"
    payload = f"{respons2['objects']}"
    payload = payload.replace('\'', '\"')
    headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
    print(f"Hentar datafil nr. {n}, ", end="")
    r4 = requests.request("POST",requesturl, headers=headers, data=payload)
    if r4.status_code == 200:
        # print(f"Vellukka spørjing på {requesturl}")
        respons4 = r4.json()
        url = respons4['urls'][innfil]['url']
        data = requests.request("GET", url)
        no = datetime.datetime.now()
        utfil = f"{tabell}-{no.year}{no.month:02}{no.day:02}{no.hour:02}{no.minute:02}-{n}"
        open(f'{utfil}.gz', 'wb').write(data.content)
        with gzip.open(f'{utfil}.gz', 'rb') as f_in:
            with open(f'{utfil}.txt', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f" skrevet til {f'{utfil}.txt'}")
        # Fjern .gz-fila etter utpakking
        os.remove(f"{utfil}.gz")
    return f"{utfil}.txt"

In [10]:
tabell = "enrollments"

In [13]:
# with open(f"sist_oppdatert_{tabell}.txt", "r") as f_in:
#     sist_oppdatert = f_in.read()
sist_oppdatert = "2025-09-15T11:27:05Z"
requesturl = f"{base_url}/dap/query/canvas/table/{tabell}/data"
payload = '{"format": "csv", "since": \"%s\"}' %(sist_oppdatert)
headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
print(f"Sender søk til {requesturl}")
r = requests.request("POST", requesturl, headers=headers, data=payload)
if r.status_code == 200:
    respons = r.json()
    id = respons['id']
    print(f"Sjekker status på jobb {id}")
    vent = True
    while vent:
        requesturl = f"{base_url}/dap//job/{id}"
        r2 = requests.request("GET", requesturl, headers=headers)
        respons2 = r2.json()
        print(respons2)
#         print(f"Status er {respons2['status']}")
        if respons2['status'] == "complete":
            vent = False
        time.sleep(5)
else:
    print(f"Feil spørsmål, kode {r2.status_code}")
antal = len(respons2['objects'])
for i in range(antal):
    hent_filar(respons2['objects'][i]['id'], i)

Sender søk til https://api-gateway.instructure.com/dap/query/canvas/table/enrollments/data
Sjekker status på jobb a207826a-611d-4e5d-9fcf-6ad163987c9a
{'id': 'a207826a-611d-4e5d-9fcf-6ad163987c9a', 'status': 'complete', 'objects': [{'id': 'a207826a-611d-4e5d-9fcf-6ad163987c9a/part-00000-05a94b74-63c5-4415-8fb9-688dc568cf31-c000.csv.gz'}, {'id': 'a207826a-611d-4e5d-9fcf-6ad163987c9a/part-00002-05a94b74-63c5-4415-8fb9-688dc568cf31-c000.csv.gz'}], 'expires_at': '2025-09-30T14:45:18Z', 'schema_version': 1, 'since': '2025-09-15T11:27:05Z', 'until': '2025-09-29T14:03:37Z'}
Hentar datafil nr. 0,  skrevet til enrollments-202509291707-0.txt
Hentar datafil nr. 1,  skrevet til enrollments-202509291707-1.txt
